# 11a — Legacy universe adoption (verify, then record; nothing is rebuilt)

The TRAIN ∪ COCONUT universe on Drive was finalized before manifest v3. Its `universe_manifest.json` has no source counts, so older code reported it as `unknown` ("COCONUT has no candidates") although the buckets carry COCONUT provenance.

This notebook **verifies** the persisted artifacts (counts, bucket offsets, candidate keys, mass / formula index sizes, provenance consistency, the expected TRAIN / COCONUT / overlap / union counts from `validation_v2.legacy_adoption.expected_counts`). It writes the sidecar `candidates/universe/universe_manifest_v3_adopted.json` only if **every** invariant holds.

* Read-only for every scientific file: buckets, indexes and the legacy manifest are never rewritten.
* "Contents verified" and "no reproducible (recipe) build id" are reported as **separate** facts.
* Legacy Stage-A markers (`universe_work/_done/COCONUT-chunk-NNNNN.json`) are discovered for the filter-loss audit (audit only).

Run order: 10 → **11a** → 11b → 13. Equivalent script: `scripts/adopt_legacy_universe_manifest.py`.

In [ ]:
# bootstrap: Drive, repository (cloned by notebook 10), config -- nothing is computed yet
import os, sys, json
from pathlib import Path
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_ROOT = Path(os.environ.setdefault('ENVEDA_DRIVE_ROOT', '/content/drive/MyDrive/EnvedaCASMI'))
_repo_candidates = [os.environ.get('ENVEDA_REPO_ROOT'), str(Path('/content', 'Enveda')), str(DRIVE_ROOT / 'repo')]
REPO_ROOT = next((Path(p) for p in _repo_candidates if p and (Path(p) / 'src' / 'casmi').is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'CASMI repository not found in {_repo_candidates}: run notebook 10 or set ENVEDA_REPO_ROOT')
os.environ['ENVEDA_REPO_ROOT'] = str(REPO_ROOT)
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
from casmi.workspace.config import load_v2_config
CFG, P = load_v2_config(REPO_ROOT / 'configs' / 'casmi_v2_colab.yaml')
P.ensure()
NOTEBOOK = '11a_colab_legacy_universe_adoption'
UNIVERSE_ROOT = P.candidate_db_dir / 'universe'
WORK = P.candidate_db_dir / 'universe_work'
EXPECTED = (CFG['validation_v2'].get('legacy_adoption') or {}).get('expected_counts') or {}
ADOPT = True          # False: verify and report only (dry run)
print('universe', UNIVERSE_ROOT, '| legacy work', WORK, '| expected counts', EXPECTED)

In [ ]:
# current universe identity + Stage-A discovery (modern ACTIVE pointers AND legacy markers)
import pandas as pd
from casmi.candidates.legacy_adoption import discover_stage_a_builds, effective_universe_identity
UID_BEFORE = effective_universe_identity(UNIVERSE_ROOT)
print(json.dumps({k: UID_BEFORE.get(k) for k in ('kind', 'build_id', 'universe_status', 'external_sources', 'problems')}, indent=2))
DISC = discover_stage_a_builds(WORK)
rows = [{k: r.get(k) for k in ('source', 'legacy', 'status', 'build_id', 'n_chunks', 'n_input', 'n_kept', 'n_filtered', 'n_rejected')}
        for recs in DISC.values() for r in recs]
display(pd.DataFrame(rows))
if not rows:
    print('no Stage-A builds found under', WORK, '(neither stage_a/ACTIVE pointers nor legacy _done/<SRC>-chunk-*.json markers)')

In [ ]:
# verification from persisted artifacts (one pyarrow pass over the buckets; no RDKit)
from casmi.candidates.legacy_adoption import verify_legacy_universe
VER = verify_legacy_universe(UNIVERSE_ROOT, expected_counts=EXPECTED, work_root=WORK)
display(pd.DataFrame(VER['checks']))
print('status            :', VER['status'])
print('contents_verified :', VER['contents_verified'])
print('counts            :', json.dumps(VER.get('counts'), indent=2))
print('per_source        :', VER.get('per_source'))
print('overlap_counts    :', VER.get('overlap_counts'))

In [ ]:
# write the sidecar only if every invariant holds (refused otherwise); idempotent for identical contents
from casmi.candidates.legacy_adoption import LegacyAdoptionRefused, adopt_legacy_universe
try:
    RES = adopt_legacy_universe(UNIVERSE_ROOT, expected_counts=EXPECTED, work_root=WORK, write=ADOPT)
    ADOPTION_STATUS = RES['status']
    print('adoption:', ADOPTION_STATUS, '| written:', RES['written'], '| sidecar:', RES['path'])
    if RES['adopted_manifest']:
        print('adopted_build_id:', RES['adopted_manifest']['adopted_build_id'])
        print('reproducible_build_id_available:', RES['adopted_manifest']['reproducible_build_id_available'])
except LegacyAdoptionRefused as e:
    RES, ADOPTION_STATUS = None, 'ADOPTION_REFUSED'
    print('ADOPTION_REFUSED:', e)
UID_AFTER = effective_universe_identity(UNIVERSE_ROOT)
print(json.dumps({k: UID_AFTER.get(k) for k in ('kind', 'build_id', 'universe_status', 'external_sources', 'contents_verified',
                                                'reproducible_build_id_available', 'problems')}, indent=2))

In [ ]:
# adoption report on Drive (reports/universe/legacy_adoption_report.json)
from datetime import datetime, timezone
from casmi.validation.external_coverage import write_json
OUT = P.reports_dir / 'universe' / 'legacy_adoption_report.json'
write_json(OUT, {'notebook': NOTEBOOK, 'adoption_status': ADOPTION_STATUS, 'identity_before': UID_BEFORE, 'identity_after': UID_AFTER,
                 'verification': {k: v for k, v in VER.items() if k != 'legacy_manifest'}, 'stage_a_discovery': DISC,
                 'written_at': datetime.now(timezone.utc).isoformat()})
print('saved', OUT)
print('next: 11b_colab_validation_regimes_v2' if UID_AFTER['kind'] in ('modern', 'legacy_adopted') else
      'STOP: the universe is not adoptable -- read the failed checks above')

## Next

* `kind = legacy_adopted` → run **11b_colab_validation_regimes_v2** (regimes bind to the adopted build id), then **13_candidate_coverage**.
* `ADOPTION_REFUSED` → do not continue. A failed invariant means the persisted universe is inconsistent; inspect the failed checks. Rebuilding (notebook 12) is a deliberate user decision.